In [ ]:
# 本镜像由 AutoDL 用户 O_O 制作，未经许可禁止二次发布！
# QQ交流群：见个人主页(https://www.codewithgpu.com/u/O_O)左侧简介

## 0、准备

In [1]:

import os
import torch

from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

model_path = "/root/models/Qwen2.5-VL-7B-Instruct/" #"Qwen/Qwen2.5-VL-7B-Instruct"
assert os.path.exists(model_path)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="flash_attention_2",
)

#模型中每幅图像的视觉标记数量的默认范围为 4-16384。
#您可以根据需要设置 min_pixels 和 max_pixels，例如 256-1280 的 token 范围，以平衡性能和显存占用
processor = AutoProcessor.from_pretrained(
    model_path,
    # min_pixels=256*28*28, max_pixels=1280*28*28,
)

print('\n模型加载完毕！')

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]


模型加载完毕！


## 1、基本

In [2]:

img_path = "/root/autodl-fs/demo/grounding_data/goden.jpg"

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                # 图片可以是 URL、本地图片路径、base64编码（data:image;base64......）
                "image": img_path,
            },
            {"type": "text", "text": """
Objective:
Analyze images and extract visually distinctive attributes for objects belonging to these categories:
[  "Labrador Retriever""]

Attribute Priority (0-1 weight score, 0=exclude, 1=definitive):
Distinctive Visual Features (e.g., ribbed bottle, stemmed wine glass)
length of the fur(e.g., long,short)

Rules:
Omit categories not present in the image.
Only include attributes verifiable by visual cues (no inference).
For each object in the category list, must output attributes ()
For unrecognizable objects, use [Object Category]: [unknown (0)] with at least 1 attribute
Each object must list at least 5 attributes (inferred attributes allowed with weight ≤0.4)

Output Format:
[Object Category]: [Attribute 1 (weight)], [Attribute 2 (weight)], ...

             
             """},
        ],
    }
]

# Preparation for inference
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to(model.device)

# Inference: Generation of the output
generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)

print('\n回答：\n', output_text)


/root/miniconda3/lib/python3.12/site-packages/torch/nn/modules/conv.py:605: UserWarning: Plan failed with a cudnnException: CUDNN_BACKEND_EXECUTION_PLAN_DESCRIPTOR: cudnnFinalize Descriptor Failed cudnn_status: CUDNN_STATUS_NOT_SUPPORTED (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:919.)
  return F.conv3d(



回答：
 ['[Labrador Retriever]: [long fur (1)], [golden coat color (1)], [standing posture (1)], [open mouth showing tongue (1)], [outdoor setting (1)]']


In [12]:
import os
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

model_path = "/root/models/Qwen2.5-VL-7B-Instruct/"

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto",
    use_flash_attention_2=False
)
processor = AutoProcessor.from_pretrained(model_path)

img_path = "/root/autodl-fs/demo/grounding_data/dog2.jpg"

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": img_path},
            {"type": "text", "text": """
Analyze the image thoroughly to identify all visible object categories, including both prominent and subtle elements. 
Consider the entire scene context (e.g., indoor/outdoor, setting) to ensure no objects are overlooked. 
Include all tangible items that can be visually confirmed, regardless of size or position.

Strict output format:
category1, category2, category3, ...

Rules:
1. Use only concise, singular nouns (e.g., chair, book, tree)
2. Separate categories with commas (no conjunctions like "and")
3. Exclude abstract concepts or actions (only physical objects)
4. Ensure complete coverage (no missing recognizable objects)
5. No additional text, explanations, or formatting beyond the required line
            """}
        ],
    }
]

# 推理准备
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to(model.device)

# 推理生成
generated_ids = model.generate(** inputs, max_new_tokens=256)  # 增加生成长度以容纳更多物体
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)

print(output_text[0])


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

grass, dogs, shadows


In [8]:
import os
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

model_path = "/root/models/Qwen2.5-VL-7B-Instruct/"

# 关键修改：禁用 FlashAttention
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto",
    use_flash_attention_2=False  # 适配旧款 GPU
)
processor = AutoProcessor.from_pretrained(model_path)

img_path = "/root/autodl-fs/demo/grounding_data/annotated_000000422170.jpg"

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": img_path},
            {"type": "text", "text": """
请检测图像中包含的所有物体类别，仅输出能通过视觉明确识别的物体。
输出格式必须严格遵循：
region contains: 类别1, 类别2, 类别3, ...

注意：
1. 用英文逗号分隔类别
2. 不添加任何额外解释或说明
3. 确保包含所有可识别的物体
            """}
        ],
    }
]

# 推理准备
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to(model.device)

# 推理生成
generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)

print(output_text[0])

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

region contains: cell phone, bottle, handbag


## 2、多图推理

In [3]:

img1_path = '/root/assets/p1.jpg'
img2_path = '/root/assets/p2.jpg'

# Messages containing multiple images and a text query
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": img1_path},
            {"type": "image", "image": img2_path},
            {"type": "text", "text": "找出图片的不同之处."},
        ],
    }
]

# Preparation for inference
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to("cuda")

# Inference
generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print('\n回答：\n', output_text)


/root/miniconda3/lib/python3.12/site-packages/torch/nn/modules/conv.py:605: UserWarning: Plan failed with a cudnnException: CUDNN_BACKEND_EXECUTION_PLAN_DESCRIPTOR: cudnnFinalize Descriptor Failed cudnn_status: CUDNN_STATUS_NOT_SUPPORTED (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:919.)
  return F.conv3d(



回答：
 ['这两张图片的主要不同之处在于狗的表情。在第一张图片中，狗的嘴巴是张开的，露出舌头，看起来像是在笑或叫喊；而在第二张图片中，狗的嘴巴闭合，眼睛也显得比较平静，整体表情更加温和。除此之外，两张图片中的其他元素（如老鼠、船、背景等）都是相同的。']


## 3、视频推理

In [6]:

# Messages containing a images list as a video and a text query
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "video",
                "video": [
                    "file:///path/to/frame1.jpg",
                    "file:///path/to/frame2.jpg",
                    "file:///path/to/frame3.jpg",
                    "file:///path/to/frame4.jpg",
                ],
            },
            {"type": "text", "text": "Describe this video."},
        ],
    }
]

# Messages containing a video url and a text query
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "video",
                "video": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen2-VL/space_woaudio.mp4",
            },
            {"type": "text", "text": "Describe this video."},
        ],
    }
]

# Messages containing a local video path and a text query
fps = 8.0
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "video",
                "video": "/root/assets/panda.mp4",
                "max_pixels": 720 * 720,
                "fps": fps,
            },
            {"type": "text", "text": "描述这个视频"},
        ],
    }
]

# Preparation for inference
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs, video_kwargs = process_vision_info(messages, return_video_kwargs=True)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    # fps=fps,
    padding=True,
    return_tensors="pt",
    **video_kwargs,
)
inputs = inputs.to("cuda")

# Inference
generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)

print('\n回答：\n', output_text)



回答：
 ['这段视频展示了两只大熊猫在户外的场景。它们背对着镜头，坐在草地上，似乎在享受周围的环境。背景是一片绿色的草地和一些植物，显得非常自然和宁静。大熊猫的毛色是典型的黑白相间，看起来非常可爱。视频左上角有一个熊猫标志和“iPanda”的字样，右下角有中文文字说明拍摄地点为中国大熊猫保护研究中心。整体画面给人一种温馨和谐的感觉。']


## 4、批量推理

In [8]:

img1_path = '/root/assets/p1.jpg'
img2_path = '/root/assets/p2.jpg'

# Sample messages for batch inference
messages1 = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": img1_path},
            {"type": "image", "image": img2_path},
            {"type": "text", "text": "图片中有哪些相同元素？"},
        ],
    }
]
messages2 = [
    {"role": "system", "content": "你是一个好助手！"},
    {"role": "user", "content": "你是谁？"},
]
# Combine messages for batch processing
messages = [messages1, messages2]

# Preparation for batch inference
texts = [
    processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
    for msg in messages
]
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=texts,
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    padding_side='left',
    return_tensors="pt",
)
inputs = inputs.to("cuda")

# Batch Inference
generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_texts = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print('\n回答：\n', output_texts)


回答：
 ['这两张图片中的相同元素包括：\n\n1. **场景**：两张图片都描绘了两只动物（一只狗和一只老鼠）在一条小船上。\n2. **背景**：背景都是蓝天白云，太阳挂在左上角，水面上有波浪。\n3. **船**：两幅图中的船是一样的木制小船。\n4. **动作**：老鼠站在船头，似乎在跳舞或挥手，而狗坐在船尾，面带微笑。\n\n主要的不同在于狗的表情和嘴巴的状态。第一张图片中的狗嘴巴是张开的，露出舌头；而在第二张图片', '我是来自阿里云的大规模语言模型，我叫通义千问。']


####制作数据集

In [ ]:
import json
import os
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
from PIL import Image

# ===================== 断点续传工具函数 =====================
def load_progress(progress_file):
    """读取进度文件，返回已处理的行数"""
    if os.path.exists(progress_file):
        with open(progress_file, 'r', encoding='utf-8') as f:
            try:
                return int(f.read().strip())
            except:
                return 0
    return 0

def save_progress(progress_file, processed_lines):
    """保存已处理的行数到进度文件"""
    with open(progress_file, 'w', encoding='utf-8') as f:
        f.write(str(processed_lines))

# ===================== 初始化Qwen2.5-VL模型 =====================
def init_qwen_model(model_path):
    print("正在加载Qwen2.5-VL模型...")
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        model_path,
        torch_dtype=torch.float16,
        device_map="auto",
        use_flash_attention_2=False
    )
    processor = AutoProcessor.from_pretrained(model_path)
    print("模型加载完成！")
    return model, processor

# ===================== 核心功能：裁剪region并推理 =====================
def infer_region_categories(image_path, bbox_list, model, processor):
    try:
        img_pil = Image.open(image_path).convert("RGB")
    except FileNotFoundError:
        raise FileNotFoundError(f"原始图片未找到：{image_path}")
    except Exception as e:
        raise Exception(f"读取图片失败：{str(e)}")
    
    img_width, img_height = img_pil.size
    region_imgs = []
    for bbox in bbox_list:
        x1, y1, x2, y2 = map(int, bbox[:4])
        x1 = max(0, x1)
        y1 = max(0, y1)
        x2 = min(img_width, x2)
        y2 = min(img_height, y2)
        if x1 >= x2 or y1 >= y2:
            print(f"  警告：无效bbox [{x1},{y1},{x2},{y2}]，跳过该区域")
            continue
        cropped_img = img_pil.crop((x1, y1, x2, y2))
        region_imgs.append(cropped_img)

    if not region_imgs:
        return "no_valid_regions"

    messages = [
        {
            "role": "user",
            "content": [
                *[{"type": "image", "image": img} for img in region_imgs],
                {"type": "text", "text": """
These are cropped regions from the same image. Identify all visible object categories in these regions.
Strict rules:
1. Use only categories from the LVIS dataset (no custom categories).
2. Discard uncertain categories (no guessing).
3. Output format: category1,category2,category3
4. Singular nouns, separated by English commas (no spaces).
5. Only physical objects, no abstract concepts.
6. No extra text, only the category list.
                """}
            ],
        }
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )
    inputs = inputs.to(model.device)

    generated_ids = model.generate(** inputs, max_new_tokens=64, temperature=0.1, do_sample=False)
    generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=True
    )[0]

    output_text = output_text.strip().replace(" ", "")
    if not output_text:
        return "no_identifiable_objects"
    return output_text

# ===================== JSONL处理主函数（支持断点续传） =====================
def process_jsonl_with_inference(file_path, output_jsonl_path, progress_file, image_root, model, processor, max_lines=None):
    processed_data = []
    output_dir = os.path.dirname(output_jsonl_path)
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # 加载已处理进度
    start_line = load_progress(progress_file)
    print(f"已处理 {start_line} 行，将从第 {start_line+1} 行开始...")

    with open(file_path, 'r', encoding='utf-8') as f_in, open(output_jsonl_path, 'a', encoding='utf-8') as f_out:
        # 跳过已处理的行
        for _ in range(start_line):
            next(f_in, None)  # 无更多行时返回None，避免报错
        
        line_count = start_line  # 从已处理行数开始计数
        for line in f_in:
            # 限制最大处理行数（None表示处理全部）
            if max_lines is not None and line_count >= max_lines:
                break
            
            try:
                data = json.loads(line.strip())
                if not all(key in data for key in ["filename", "grounding"]) or "regions" not in data["grounding"]:
                    print(f"跳过第 {line_count+1} 行：缺少必要字段")
                    line_count += 1
                    save_progress(progress_file, line_count)  # 即使跳过也更新进度
                    continue
                
                image_path = os.path.join(image_root, data["filename"])
                if not os.path.exists(image_path):
                    print(f"跳过第 {line_count+1} 行：图片未找到 {image_path}")
                    line_count += 1
                    save_progress(progress_file, line_count)
                    continue
                
                print(f"\n处理第 {line_count+1} 行：{data['filename']} → 共 {len(data['grounding']['regions'])} 个region")
                for idx, region in enumerate(data["grounding"]["regions"], 1):
                    bbox_list = region["bbox"] if isinstance(region["bbox"][0], list) else [region["bbox"]]
                    # print(f"  正在推理region {idx}/{len(data['grounding']['regions'])}...")
                    
                    try:
                        categories = infer_region_categories(image_path, bbox_list, model, processor)
                        region["region_contains"] = f"region contains: {categories}"
                        # print(f"  region {idx} 推理结果：{region['region_contains']}")
                    except Exception as e:
                        region["region_contains"] = "region contains: inference_failed"
                        print(f"  region {idx} 推理失败：{str(e)}")
                
                # 追加写入（不是覆盖），避免丢失已处理数据
                json.dump(data, f_out, ensure_ascii=False)
                f_out.write('\n')
                processed_data.append(data)
                line_count += 1
                save_progress(progress_file, line_count)  # 每处理完一行更新进度
                
            except json.JSONDecodeError as e:
                print(f"跳过第 {line_count+1} 行：JSON解析错误 - {str(e)}")
                line_count += 1
                save_progress(progress_file, line_count)
            except Exception as e:
                print(f"跳过第 {line_count+1} 行：处理错误 - {str(e)}")
                line_count += 1
                save_progress(progress_file, line_count)
    
    return processed_data

# ===================== 配置参数 =====================
input_jsonl = "./grounding_data/coco/annotations/instances_train2017_vg_merged6.jsonl"
output_jsonl = "./grounding_data/coco/annotations/processed_instances_train2017_vg_merged6.jsonl"
progress_file = "./grounding_data/coco/annotations/process_progress.txt"  # 进度文件路径
image_root = "./grounding_data/coco/train2017"
model_path = "/root/models/Qwen2.5-VL-7B-Instruct/"
max_process_lines = None  # None=处理全部记录，可改为具体数值（如100）

# ===================== 执行全流程 =====================
if __name__ == "__main__":
    # 检查PIL
    try:
        from PIL import Image
    except ImportError:
        print("未找到PIL库，正在尝试安装...")
        import subprocess
        import sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "pillow", "--upgrade"])
        from PIL import Image
        print("PIL库安装成功！")
    
    print("="*60)
    print("开始JSONL处理+Qwen2.5-VL区域类别推理（支持断点续传）")
    print("="*60)
    
    model, processor = init_qwen_model(model_path)
    
    processed_data = process_jsonl_with_inference(
        file_path=input_jsonl,
        output_jsonl_path=output_jsonl,
        progress_file=progress_file,
        image_root=image_root,
        model=model,
        processor=processor,
        max_lines=max_process_lines
    )
    
    print(f"\n" + "="*60)
    print("处理完成！")
    print(f"输入文件：{input_jsonl}")
    print(f"输出文件：{output_jsonl}")
    print(f"成功处理记录数：{len(processed_data)}")
    print("="*60)

# 用于处理清理代码当中的,region contains的混乱局面,剔除无关类,剔除相似易混淆类

In [ ]:
import json
import os
import torch
from collections import defaultdict
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

# ===================== 断点续传工具函数 =====================
def load_progress(progress_file):
    if os.path.exists(progress_file):
        with open(progress_file, 'r', encoding='utf-8') as f:
            try:
                return int(f.read().strip())
            except:
                return 0
    return 0

def save_progress(progress_file, processed_lines):
    with open(progress_file, 'w', encoding='utf-8') as f:
        f.write(str(processed_lines))

# ===================== 初始化Qwen2.5-VL模型 =====================
def init_qwen_model(model_path):
    print("正在加载Qwen2.5-VL模型用于二次清洗...")
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        model_path,
        torch_dtype=torch.float16,
        device_map="auto",
        use_flash_attention_2=False
    )
    processor = AutoProcessor.from_pretrained(model_path)
    print("模型加载完成！")
    return model, processor

# ===================== 核心：大模型语义过滤函数 =====================
def filter_by_qwen(phrase, raw_categories, model, processor):
    """
    使用Qwen2.5-VL判断子类别与主物体的语义关联性，返回相关类别
    :param phrase: 主物体（如wine glass）
    :param raw_categories: 原始region_contains的类别列表（如[glass,handle,chair]）
    :param model: 加载好的Qwen2.5-VL模型
    :param processor: 模型对应的processor
    :return: 过滤后的类别列表（去重）
    """
    # 无原始类别时直接返回空
    if not raw_categories:
        return []
    
    # 构造Prompt：明确任务+约束输出格式
    prompt_text = f"""
任务：判断以下子类别是否与主物体【{phrase}】存在直接语义关联（如部件关系、属性关系、携带/附属关系）。
子类别列表：{','.join(raw_categories)}
严格遵守以下规则：
1. 仅保留与主物体**强关联**的子类别，剔除空间相邻但语义无关的类别（如dining table和chair仅相邻，无关联）；
2. 剔除泛化概念（如container）、抽象概念、无意义类别；
3. 输出格式：仅返回相关类别的英文名称，用英文逗号分隔，无空格，无其他文本；
4. 对重复的类别自动去重。
    """.strip()

    # 构建模型输入（仅文本输入，无需图像）
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt_text}
            ]
        }
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(
        text=[text],
        padding=True,
        return_tensors="pt"
    )
    inputs = inputs.to(model.device)

    # 模型推理（低temperature保证输出稳定）
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=64,
        temperature=0.1,
        do_sample=False,
        num_beams=1
    )
    generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=True
    )[0].strip()

    # 解析输出结果并去重
    if not output_text:
        return []
    filtered_list = [cat.strip().lower() for cat in output_text.split(",") if cat.strip()]
    filtered_list = list(set(filtered_list))  # 最终去重
    return filtered_list

# ===================== 提取原始region_contains的类别列表 =====================
def extract_raw_categories(region_contains_str):
    """
    从"region contains: a,b,c"格式中提取原始类别列表
    """
    if not region_contains_str or "region contains:" not in region_contains_str:
        return []
    raw_part = region_contains_str.split("region contains:")[-1].strip()
    if not raw_part:
        return []
    return [cat.strip().lower() for cat in raw_part.split(",") if cat.strip()]

# ===================== 批量二次清洗JSONL =====================
def batch_second_clean(
    input_jsonl, output_jsonl, progress_file, model_path,
    max_lines=None
):
    # 初始化模型
    model, processor = init_qwen_model(model_path)
    
    # 断点续传
    start_line = load_progress(progress_file)
    print(f"二次清洗：已处理 {start_line} 行，从第 {start_line+1} 行开始...")
    
    output_dir = os.path.dirname(output_jsonl)
    if not output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    processed_count = 0
    with open(input_jsonl, 'r', encoding='utf-8') as f_in, \
         open(output_jsonl, 'a', encoding='utf-8') as f_out:
        
        # 跳过已处理行
        for _ in range(start_line):
            next(f_in, None)
        
        line_count = start_line
        for line in f_in:
            if max_lines is not None and line_count >= max_lines:
                break
            
            try:
                data = json.loads(line.strip())
                # 检查必要字段
                if "grounding" not in data or "regions" not in data["grounding"]:
                    print(f"跳过第 {line_count+1} 行：缺少grounding/regions字段")
                    line_count += 1
                    save_progress(progress_file, line_count)
                    continue
                
                # 遍历每个region做二次清洗
                for region in data["grounding"]["regions"]:
                    if "phrase" not in region or "region_contains" not in region:
                        continue
                    
                    phrase = region["phrase"]
                    raw_rc_str = region["region_contains"]
                    
                    # 步骤1：提取原始类别列表
                    raw_categories = extract_raw_categories(raw_rc_str)
                    if not raw_categories:
                        region["region_contains_cleaned"] = "region contains: "
                        continue
                    
                    # 步骤2：大模型语义过滤
                    filtered_categories = filter_by_qwen(phrase, raw_categories, model, processor)
                    
                    # 步骤3：更新清洗后字段（保留原始字段，新增cleaned字段）
                    if filtered_categories:
                        region["region_contains_cleaned"] = f"region contains: {','.join(filtered_categories)}"
                    else:
                        region["region_contains_cleaned"] = "region contains: "
                
                # 写入清洗后的数据
                json.dump(data, f_out, ensure_ascii=False)
                f_out.write('\n')
                processed_count += 1
                line_count += 1
                save_progress(progress_file, line_count)
                
                if line_count % 5 == 0:
                    print(f"已处理 {line_count} 行，成功清洗 {processed_count} 条记录")
            
            except json.JSONDecodeError:
                print(f"跳过第 {line_count+1} 行：JSON解析错误")
                line_count += 1
                save_progress(progress_file, line_count)
            except Exception as e:
                print(f"跳过第 {line_count+1} 行：处理失败 - {str(e)}")
                line_count += 1
                save_progress(progress_file, line_count)
    
    print(f"\n二次清洗完成！")
    print(f"输入文件：{input_jsonl}")
    print(f"输出文件：{output_jsonl}")
    print(f"成功清洗记录数：{processed_count}")
    return processed_count

# ===================== 配置参数 =====================
if __name__ == "__main__":
    # 路径配置
    INPUT_JSONL = "./grounding_data/coco/annotations/processed_instances_train2017_vg_merged6.jsonl"  # 第一次生成后的文件
    OUTPUT_JSONL = "./grounding_data/coco/annotations/second_cleaned_instances_train2017_vg_merged6.jsonl"  # 二次清洗后文件
    PROGRESS_FILE = "./grounding_data/coco/annotations/second_clean_progress.txt"  # 二次清洗进度文件
    MODEL_PATH = "/root/models/Qwen2.5-VL-7B-Instruct/"  # 模型路径
    MAX_LINES = None  # None处理全部，可设为100测试
    
    # 执行二次清洗
    batch_second_clean(
        input_jsonl=INPUT_JSONL,
        output_jsonl=OUTPUT_JSONL,
        progress_file=PROGRESS_FILE,
        model_path=MODEL_PATH,
        max_lines=MAX_LINES
    )

    # 单条测试示例
    test_model, test_processor = init_qwen_model(MODEL_PATH)
    test_phrase = "person"
    test_raw_cats = ["bottle", "glass", "table", "napkin", "chair", "blackboard"]
    test_filtered = filter_by_qwen(test_phrase, test_raw_cats, test_model, test_processor)
    print(f"\n测试示例：")
    print(f"主物体：{test_phrase}")
    print(f"原始类别：{test_raw_cats}")
    print(f"清洗后类别：{test_filtered}")

In [4]:
################调试cleand

In [13]:
# 调试Prompt单条Demo（精准版）
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor


# 优化后Prompt（核心：明确规则+反例+强制排除）
def debug_filter_prompt(phrase, raw_categories):
    prompt = f"""
### 任务说明
仅保留与主物体【{phrase}】存在**直接语义关联**的子类别，其他类别均需排除
1. 有效关联类型（仅保留以下3类）：
   - 部件关系：子类别是主物体的核心组成部分（如wine glass ↔ handle）；
   - 属性关系：子类别是主物体的核心属性（如bottle ↔ liquid）；
   - 携带/穿戴/使用关系：子类别是主物体（人/物）直接携带/穿戴/使用的物品（如person ↔ ring、dining table ↔ napkin）；


### 输入与输出
子类别列表：{','.join(raw_categories)}
输出要求：仅返回符合规则的英文类别，用英文逗号分隔，无空格、无额外文本，重复类别自动去重。,宁可多排除,也不少排除,最多5个子类别

### 典型反例（必须参考纠正）
❌ 错误案例1：主物体=chair，原始类别=['chair']，错误过滤结果=['foot','backrest','armrest','seat','cushion']
   ✅ 正确结果：[]（chair无关联子类别，仅自身时输出为空）
❌ 错误案例2：主物体=wine glass，原始类别=['glass','handle','container','chair']，错误过滤结果=['handle','container']
   ✅ 正确结果：['glass','handle']（必须剔除相似名称container，保留glass和handle）
###
    """.strip()
    
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], padding=True, return_tensors="pt").to(model.device)
    
    generated_ids = model.generate(**inputs, max_new_tokens=64, temperature=0.0, do_sample=False)
    output = processor.batch_decode(generated_ids[:, len(inputs.input_ids[0]):], skip_special_tokens=True)[0].strip()
    return [cat.strip().lower() for cat in output.split(",") if cat.strip()]

# 测试案例（复用原始数据中的典型场景）
test_cases = [
    ("dining table", ["glass", "bottle", "napkin", "table", "ring", "chair", "hand"]),
    ("person", ["bottle", "glass", "table", "napkin", "chair", "blackboard"]),
    ("wine glass", ["glass", "handle", "container", "chair"]),
      ("bicycle", ["bicycle","shorts","flip-flops","pedal","handlebar","brake","seat","shoe","curb","brick","hand"]),
    ("person", ["helmet","bicycle","backpack","shorts","shoes","streetlight","car","building","road","grass","plant","flowerpot","flower","umbrella","window","door","signboard","streetlight","car","building","road","grass","plant","flowerpot","flower","umbrella","window"]),
    ("backpack", ["backpack","shirt","watch"]),
    ("chair", ["chair"])
]

# 运行Demo并输出结果
for phrase, raw_cats in test_cases:
    filtered = debug_filter_prompt(phrase, raw_cats)
    print(f"主物体：{phrase}")
    print(f"原始类别：{raw_cats}")
    print(f"过滤结果：{filtered}")
    print("-" * 60)

主物体：dining table
原始类别：['glass', 'bottle', 'napkin', 'table', 'ring', 'chair', 'hand']
过滤结果：['dining table']
------------------------------------------------------------
主物体：person
原始类别：['bottle', 'glass', 'table', 'napkin', 'chair', 'blackboard']
过滤结果：['person', 'bottle', 'glass', 'napkin', 'table']
------------------------------------------------------------
主物体：wine glass
原始类别：['glass', 'handle', 'container', 'chair']
过滤结果：["['glass'", "'handle']"]
------------------------------------------------------------
主物体：bicycle
原始类别：['bicycle', 'shorts', 'flip-flops', 'pedal', 'handlebar', 'brake', 'seat', 'shoe', 'curb', 'brick', 'hand']
过滤结果：['bicycle', 'pedal', 'handlebar', 'brake', 'seat']
------------------------------------------------------------
主物体：person
原始类别：['helmet', 'bicycle', 'backpack', 'shorts', 'shoes', 'streetlight', 'car', 'building', 'road', 'grass', 'plant', 'flowerpot', 'flower', 'umbrella', 'window', 'door', 'signboard', 'streetlight', 'car', 'building', 'road', 'gras

In [8]:
# 第一步：导入必要依赖（仅需PyTorch和基础库）
import torch
import os
import logging
from collections import defaultdict

# 第二步：模拟 logger（适配 Jupyter 输出）
# 替代 mmengine 的 MMLogger，直接用 Python 原生日志输出到 Notebook
logger = logging.getLogger('test')
logger.setLevel(logging.INFO)
# 确保日志能在 Notebook 中正常打印
if not logger.handlers:
    handler = logging.StreamHandler()
    formatter = logging.Formatter('%(message)s')
    handler.setFormatter(formatter)
    logger.addHandler(handler)

# 第三步：构造模拟的 results 数据（和真实格式一致）
# 模拟按类别分组的 results：key=类别ID，value=该类别的预测框列表
results = defaultdict(list)

# 构造类别1的检测结果
results[1].extend([
    {'image_id': 1001, 'category_id': 1, 'bbox': [10, 20, 30, 40], 'score': 0.95},
    {'image_id': 1002, 'category_id': 1, 'bbox': [50, 60, 70, 80], 'score': 0.88},
    {'image_id': 1003, 'category_id': 1, 'bbox': [90, 100, 110, 120], 'score': 0.76}
])

# 构造类别2的检测结果
results[2].extend([
    {'image_id': 1001, 'category_id': 2, 'bbox': [100, 200, 300, 400], 'score': 0.75},
    {'image_id': 1004, 'category_id': 2, 'bbox': [500, 600, 700, 800], 'score': 0.69}
])

# 第四步：执行存储逻辑（和你代码中的存储部分完全一致）
# 1. 定义存储路径（Jupyter 中默认存到当前工作目录）
results_save_path = './lvis_test_results.pt'
# 2. 确保存储目录存在（如果路径包含子目录也能自动创建）
os.makedirs(os.path.dirname(results_save_path), exist_ok=True)

# 3. 存储 results
try:
    # 转换为普通 dict 避免 defaultdict 序列化问题
    torch.save(dict(results), results_save_path)
    logger.info(f'✅ 原始results已存储至：{results_save_path}')
    # 打印 Jupyter 中的文件路径，方便查看
    logger.info(f'当前工作目录：{os.getcwd()}')
except Exception as e:
    logger.warning(f'❌ 存储results失败：{str(e)}')

# 第五步：验证存储结果（加载并检查）
try:
    # 加载存储的文件
    loaded_results = torch.load(results_save_path)
    logger.info(f'\n✅ 成功加载存储的results！')
    # 检查数据完整性
    logger.info(f'存储的类别数量：{len(loaded_results.keys())}')
    logger.info(f'类别1的检测框数量：{len(loaded_results[1])}')
    logger.info(f'类别1第一个检测框的分数：{loaded_results[1][0]["score"]}')
    logger.info(f'类别2第二个检测框的bbox：{loaded_results[2][1]["bbox"]}')
except Exception as e:
    logger.warning(f'❌ 加载存储的results失败：{str(e)}')

✅ 原始results已存储至：./lvis_test_results.pt
当前工作目录：/autodl-fs/data/demo

✅ 成功加载存储的results！
存储的类别数量：2
类别1的检测框数量：3
类别1第一个检测框的分数：0.95
类别2第二个检测框的bbox：[500, 600, 700, 800]


In [2]:
import os
import torch
import numpy as np
from mmengine.logging import MMLogger
from lvis import LVIS  # 需安装 lvis-api: pip install lvis-api

# ===================== 配置项（根据你的环境修改） =====================
PT_FILE_PATH = './lvis_test_results.pt'  # 你保存的pt文件路径
LVIS_ANN_FILE = '../grounding_data/coco/annotations/lvis_v1_minival_inserted_image_name.json'  # LVIS标注文件路径
TOPK = 10000  # 原代码中的self.topk，默认10000
# =====================================================================

# 模拟LVISResults类（简化版，兼容原代码逻辑）
class LVISResults:
    def __init__(self, lvis_api, results, max_dets=-1):
        self.lvis_api = lvis_api
        self.results = results
        self.max_dets = max_dets

# 模拟LVISEval类（核心评估逻辑，简化版）
class LVISEval:
    def __init__(self, lvis_api, results, iou_type='bbox'):
        self.lvis_api = lvis_api
        self.results = results
        self.iou_type = iou_type
        self.params = self.lvis_api.get_eval_params()
        self.results = {}

    def run(self):
        # 调用LVIS官方评估逻辑
        self.eval_results = self.lvis_api.evaluate(self.results.results, self.params, self.iou_type)

    def print_results(self):
        # 打印评估结果
        for metric, value in self.eval_results.items():
            if metric.startswith('AP'):
                print(f'{metric}: {value:.4f}')

    @property
    def results(self):
        return self.eval_results

# 核心评估函数（独立版）
def compute_metrics_from_pt(results: dict, lvis_api, topk=TOPK):
    # 初始化日志
    logger = MMLogger.get_current_instance() if MMLogger.has_instance() else MMLogger('eval_from_pt')

    new_results = []
    missing_dets_cats = set()

    # 处理按类别分组的结果
    for cat, cat_anns in results.items():
        if len(cat_anns) < topk:
            missing_dets_cats.add(cat)
        # 按分数排序，取topk
        new_results.extend(
            sorted(cat_anns, key=lambda x: x['score'], reverse=True)[:topk]
        )

    # 打印缺失检测的类别提示
    if missing_dets_cats:
        logger.info(
            f'\n===\n'
            f'{len(missing_dets_cats)} classes had less than {topk} '
            f'detections!\n Outputting {topk} detections for each '
            f'class will improve AP further.\n ===')

    # 初始化评估器
    new_results_obj = LVISResults(lvis_api, new_results, max_dets=-1)
    lvis_eval = LVISEval(lvis_api, new_results_obj, iou_type='bbox')
    lvis_eval.params.max_dets = -1  # 不限制每张图的检测框数量
    lvis_eval.run()  # 执行评估
    lvis_eval.print_results()  # 打印结果

    # 提取AP相关指标
    metrics = {
        k: v for k, v in lvis_eval.results.items() if k.startswith('AP')
    }
    logger.info(f'mAP_copypaste: {metrics}')
    return metrics

# 主函数：加载pt文件并评估
def main():
    # 1. 加载pt文件
    if not os.path.exists(PT_FILE_PATH):
        raise FileNotFoundError(f'未找到pt文件：{PT_FILE_PATH}')
    results = torch.load(PT_FILE_PATH, map_location='cpu')  # 加载到CPU，避免GPU依赖
    print(f'✅ 成功加载pt文件，包含 {len(results)} 个类别的检测结果')

    # 2. 初始化LVIS API（加载标注文件）
    lvis_api = LVIS(LVIS_ANN_FILE)
    print(f'✅ 成功加载LVIS标注文件：{LVIS_ANN_FILE}')

    # 3. 执行评估
    metrics = compute_metrics_from_pt(results, lvis_api, topk=TOPK)

    # 4. 输出最终结果
    print('\n==================== 最终评估结果 ====================')
    for metric, value in metrics.items():
        print(f'{metric}: {value:.4f}')

if __name__ == '__main__':
    # 初始化日志
    MMLogger.get_instance('eval_from_pt', log_level='INFO')
    main()


CondaError: Run 'conda init' before 'conda activate'



ModuleNotFoundError: No module named 'mmengine'